In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *

@dp.table(
    name="customers",
    table_properties={"quality":"bronze"}

)
def customers_dlt():
    return spark.readStream.table("dev.dlt.customers_dlt")

In [0]:
@dp.table(
    name="orders",
    table_properties={"quality":"bronze"}

)
def customers_dlt():
    return spark.read.table("dev.dlt.orders_dlt")

In [0]:
@dp.view(
    name='customer_orders'
)
def fn_customer_orders():
    df_c = spark.read.table("customers")
    df_o = spark.read.table('orders')
    df_join = df_c.join(df_o,how="left_outer",on=df_c.c_custkey==df_o.o_custkey)
    return df_join

In [0]:
from pyspark.sql.functions import * 

@dp.table(
    name='joined_silver',
    table_properties={
        "quality":"silver"}
)
def fn_customer_orders():
    df = spark.read.table('customer_orders').withColumn("_insertDate",current_timestamp())
    return df

In [0]:
from pyspark.sql.functions import * 

@dp.table(
    table_properties={
        "quality":"gold"}
)
def orders_aggregate_gold():
    df = spark.read.table('joined_silver')
    df_final = df.groupBy('c_mktsegment').agg(count('o_orderkey').alias('count_of_orders'),sum('o_totalprice').alias('sum_totalprice')).withColumn('_insertDate',current_timestamp())
    return df_final